# 00. Setup del Proyecto — `proyecto_integrador_v2`

Este notebook inicializa la estructura del proyecto **proyecto_integrador_v2** en Google Drive.

## Objetivo del proyecto

El objetivo principal de esta versión no es solamente clasificar razas de perros, sino construir un pipeline de **re-identificación visual de perros perdidos y encontrados**.

El sistema debe permitir:

1. Recibir imágenes de perros perdidos o encontrados.
2. Detectar al perro en la imagen.
3. Generar un crop del perro.
4. Extraer un embedding visual.
5. Comparar ese embedding contra una base de datos de perros reportados.
6. Devolver posibles coincidencias visuales.
7. Usar raza, confianza, fecha y ubicación como señales auxiliares.

La raza no es la decisión principal; la decisión principal se basa en **similitud visual mediante embeddings**.

## Estructura general

La estructura base será:

```text
/content/drive/MyDrive/proyecto_integrador_v2/
│
├── raw_data/
│   ├── images/
│   ├── lost_reports/
│   ├── found_reports/
│   └── identity_test/
│
├── curated_data/
│   ├── images_curated/
│   └── quality_reports/
│
├── processed_data/
│   ├── dog_crops/
│   ├── embeddings/
│   ├── metadata/
│   ├── vector_index/
│   └── search_results/
│
├── models/
│   ├── yolo/
│   ├── embedding_model/
│   └── classifiers/
│
├── notebooks/
│
├── reports/
│
└── README.md
```

In [ ]:
# 1. Montar Google Drive


from pathlib import Path
from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 2. Definir ruta base del proyecto


from pathlib import Path
import json
from datetime import datetime

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto_integrador_v2")

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 3. Crear estructura de carpetas


folders = [
    "raw_data",
    "raw_data/images",
    "raw_data/lost_reports",
    "raw_data/found_reports",
    "raw_data/identity_test",

    "curated_data",
    "curated_data/images_curated",
    "curated_data/quality_reports",

    "processed_data",
    "processed_data/dog_crops",
    "processed_data/embeddings",
    "processed_data/metadata",
    "processed_data/vector_index",
    "processed_data/search_results",

    "models",
    "models/yolo",
    "models/embedding_model",
    "models/classifiers",

    "notebooks",

    "reports",
    "reports/figures",
    "reports/tables",
]

for folder in folders:
    path = PROJECT_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)

print("Estructura creada correctamente en:")
print(PROJECT_ROOT)

In [ ]:
# 4. Crear subestructura para reportes lost/found


example_folders = [
    "raw_data/lost_reports/dog_001",
    "raw_data/found_reports/report_001",
    "raw_data/identity_test/dog_001",
    "raw_data/identity_test/dog_002",
]

for folder in example_folders:
    path = PROJECT_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)

print("Subestructura de ejemplo creada.")

## Convención de metadata

Cada reporte de perro perdido o encontrado puede tener un archivo `metadata.json`.

Ejemplo:

```json
{
  "report_id": "lost_001",
  "dog_id": "dog_001",
  "report_type": "lost",
  "reported_at": "2026-05-28",
  "location": {
    "city": "Culiacán",
    "state": "Sinaloa",
    "country": "Mexico",
    "latitude": null,
    "longitude": null
  },
  "description": "Perro pequeño color blanco con café",
  "contact": null,
  "status": "active"
}
```

Esta metadata será útil después para combinar similitud visual con fecha, ubicación y tipo de reporte.

In [ ]:
# 5. Crear metadata.json de ejemplo


lost_metadata_example = {
    "report_id": "lost_001",
    "dog_id": "dog_001",
    "report_type": "lost",
    "reported_at": datetime.now().strftime("%Y-%m-%d"),
    "location": {
        "city": None,
        "state": None,
        "country": None,
        "latitude": None,
        "longitude": None
    },
    "description": "Ejemplo de perro perdido.",
    "contact": None,
    "status": "active"
}

found_metadata_example = {
    "report_id": "found_001",
    "dog_id": None,
    "report_type": "found",
    "reported_at": datetime.now().strftime("%Y-%m-%d"),
    "location": {
        "city": None,
        "state": None,
        "country": None,
        "latitude": None,
        "longitude": None
    },
    "description": "Ejemplo de perro encontrado.",
    "contact": None,
    "status": "active"
}

identity_metadata_example = {
    "dog_id": "dog_001",
    "notes": "Carpeta de prueba con varias fotos del mismo perro para evaluación de re-identificación.",
    "expected_use": "same_dog_evaluation"
}

metadata_files = [
    (PROJECT_ROOT / "raw_data/lost_reports/dog_001/metadata.json", lost_metadata_example),
    (PROJECT_ROOT / "raw_data/found_reports/report_001/metadata.json", found_metadata_example),
    (PROJECT_ROOT / "raw_data/identity_test/dog_001/metadata.json", identity_metadata_example),
]

for path, data in metadata_files:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("Archivos metadata.json de ejemplo creados.")

In [ ]:
# 6. Crear configuración global del proyecto


config = {
    "project_name": "proyecto_integrador_v2",
    "project_goal": "Re-identificación visual de perros perdidos y encontrados mediante embeddings",
    "main_task": "visual_reidentification",
    "secondary_task": "breed_classification_auxiliary",
    "image_size": 224,
    "detector": {
        "default_model": "yolo26s.pt",
        "fallback_model": "yolo11s.pt",
        "confidence_threshold": 0.25,
        "crop_margin": 0.15
    },
    "embedding": {
        "baseline_model": "EfficientNetB0",
        "embedding_layer_rule": "penultimate_dense_layer",
        "normalization": "l2",
        "similarity_metric": "cosine_similarity"
    },
    "decision_layer": {
        "high_confidence_similarity": 0.90,
        "possible_match_similarity": 0.80,
        "weak_match_similarity": 0.70
    },
    "created_at": datetime.now().isoformat()
}

CONFIG_PATH = PROJECT_ROOT / "project_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Configuración guardada en:", CONFIG_PATH)

In [ ]:
# 7. Crear README inicial


readme_text = '''# proyecto_integrador_v2

Este proyecto implementa un pipeline de visión computacional para apoyar la búsqueda de perros perdidos y encontrados.

## Objetivo

El objetivo principal es construir una base de búsqueda visual usando embeddings.
La clasificación de raza se utiliza como señal auxiliar, pero la decisión principal se basa en similitud visual.

## Pipeline

1. Curaduría y control de calidad de imágenes.
2. Detección del perro con YOLO.
3. Generación de crops del perro.
4. Extracción de embeddings visuales.
5. Búsqueda vectorial mediante similitud coseno.
6. Evaluación de re-identificación del mismo perro.
7. Capa de decisión para posibles coincidencias.

## Concepto principal

Cada imagen de perro se transforma en un vector visual.
Una imagen nueva puede compararse contra la base de embeddings para encontrar posibles coincidencias de perros reportados como perdidos o encontrados.
'''

README_PATH = PROJECT_ROOT / "README.md"

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README creado en:", README_PATH)

In [ ]:
# 8. Verificar estructura creada


def print_tree(path, max_depth=3, prefix=""):
    path = Path(path)

    if max_depth < 0:
        return

    items = sorted(list(path.iterdir()), key=lambda p: (p.is_file(), p.name.lower()))

    for i, item in enumerate(items):
        connector = "└── " if i == len(items) - 1 else "├── "
        print(prefix + connector + item.name)

        if item.is_dir():
            extension = "    " if i == len(items) - 1 else "│   "
            print_tree(item, max_depth=max_depth - 1, prefix=prefix + extension)

print(PROJECT_ROOT.name)
print_tree(PROJECT_ROOT, max_depth=3)